# 🔍 Semantic Search with OpenAI Embeddings + Pinecone + Cross-Encoder

Full semantic search pipeline:
- 📦 **Dataset**: SQuAD (Stanford QA) — Wikipedia passages
- 🧠 **Embeddings**: OpenAI `text-embedding-3-small` (auto-falls back to free local model)
- 🌲 **Vector DB**: Pinecone (serverless)
- ⚖️ **Reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2`

```
Query ──► Embed ──► Pinecone ANN Search ──► Top-K candidates
                                                   │
                                  ┌────────────────┘
                                  ▼
                       Cross-Encoder Reranking ──► Final Top-N
```

## Step 1 — Install Dependencies

In [ ]:
!pip install -q openai pinecone sentence-transformers datasets tqdm

## Step 2 — API Keys & Config

- OpenAI key: https://platform.openai.com/api-keys
- Pinecone key: https://app.pinecone.io → API Keys

> **No OpenAI credits?** No problem — set `FORCE_LOCAL_EMBED = True` in Step 4 and the notebook uses a free local model instead.

In [ ]:
import os
from getpass import getpass

OPENAI_API_KEY   = getpass('🔑 OpenAI API key (press Enter to skip if using local embeddings): ')
PINECONE_API_KEY = getpass('🌲 Pinecone API key: ')

PINECONE_CLOUD  = 'aws'
PINECONE_REGION = 'us-east-1'

MAX_DOCS       = 5000   # passages to index (reduce to 1000 for a quick test)
TOP_K          = 10     # candidates from Pinecone before reranking
TOP_N_RERANKED = 5      # final results after reranking

os.environ['OPENAI_API_KEY']   = OPENAI_API_KEY
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
print('✅ Config ready!')

## Step 3 — Load SQuAD Dataset

SQuAD is a well-known NLP benchmark built from Wikipedia. We extract unique context passages.

In [ ]:
from datasets import load_dataset

print('⏳ Loading SQuAD dataset...')
try:
    squad = load_dataset('rajpurkar/squad', split='train', trust_remote_code=False)
    print("✅ Loaded via 'rajpurkar/squad'")
except Exception as e:
    print(f'rajpurkar/squad failed ({e}), trying legacy name...')
    squad = load_dataset('squad', split='train')
    print("✅ Loaded via legacy 'squad'")

seen, passages = set(), []
for row in squad:
    ctx = row['context'].strip()
    if ctx not in seen:
        seen.add(ctx)
        passages.append({'id': f'doc-{len(passages)}', 'text': ctx, 'title': row['title']})
    if len(passages) >= MAX_DOCS:
        break

print(f'✅ {len(passages):,} unique passages loaded')
print(f"\nSample ({passages[0]['title']}): {passages[0]['text'][:200]}...")

## Step 4 — Embeddings (OpenAI or Free Local Fallback)

| Backend | Model | Dim | Cost |
|---|---|---|---|
| OpenAI | `text-embedding-3-small` | 1536 | ~\$0.03 for 5k passages |
| **Local (free)** | `all-MiniLM-L6-v2` | 384 | **Free, runs on Colab CPU** |

Set `FORCE_LOCAL_EMBED = True` to always use the free model.

In [ ]:
import time
from tqdm.auto import tqdm

# ── Toggle this to True to skip OpenAI entirely ──────────────────────────
FORCE_LOCAL_EMBED = False
# ─────────────────────────────────────────────────────────────────────────

USE_OPENAI = False

if not FORCE_LOCAL_EMBED and OPENAI_API_KEY:
    try:
        from openai import OpenAI
        _oa = OpenAI(api_key=OPENAI_API_KEY)
        _oa.embeddings.create(model='text-embedding-3-small', input=['ping'])  # quota probe
        USE_OPENAI = True
        EMBED_DIM  = 1536
        ACTIVE_MODEL = 'text-embedding-3-small'
        print('✅ OpenAI quota OK — using text-embedding-3-small (1536-dim)')
    except Exception as e:
        print(f'⚠️  OpenAI error ({type(e).__name__}): {e}')
        print('   → Switching to free local model.')

if not USE_OPENAI:
    from sentence_transformers import SentenceTransformer
    EMBED_DIM    = 384
    ACTIVE_MODEL = 'all-MiniLM-L6-v2'
    print(f'⏳ Loading local model {ACTIVE_MODEL}...')
    _local = SentenceTransformer(ACTIVE_MODEL)
    print('✅ Local model ready (384-dim, completely free)')

# ── Unified embedding functions ───────────────────────────────────────────
def get_embeddings(texts, batch_size=None):
    if USE_OPENAI:
        batch_size = batch_size or 100
        out = []
        for i in tqdm(range(0, len(texts), batch_size), desc='OpenAI embed'):
            r = _oa.embeddings.create(model=ACTIVE_MODEL, input=texts[i:i+batch_size])
            out.extend([x.embedding for x in r.data])
            time.sleep(0.05)
        return out
    else:
        batch_size = batch_size or 256
        out = []
        for i in tqdm(range(0, len(texts), batch_size), desc='Local embed'):
            vecs = _local.encode(texts[i:i+batch_size], normalize_embeddings=True, show_progress_bar=False)
            out.extend(vecs.tolist())
        return out

def embed_query(query):
    if USE_OPENAI:
        return _oa.embeddings.create(model=ACTIVE_MODEL, input=[query]).data[0].embedding
    else:
        return _local.encode([query], normalize_embeddings=True)[0].tolist()

# ── Pinecone index name includes model so dims never clash ────────────────
INDEX_NAME = f"semsearch-{'openai' if USE_OPENAI else 'local'}"

print(f'\n⏳ Embedding {len(passages):,} passages...')
texts      = [p['text'] for p in passages]
embeddings = get_embeddings(texts)
print(f'✅ Done — {len(embeddings):,} embeddings, dim={len(embeddings[0])}')

## Step 5 — Set Up Pinecone & Upsert Vectors

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    print(f"⏳ Creating index '{INDEX_NAME}' (dim={EMBED_DIM})...")
    pc.create_index(
        name=INDEX_NAME, dimension=EMBED_DIM, metric='cosine',
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(1)
    print('✅ Index created!')
else:
    print(f"✅ Reusing existing index '{INDEX_NAME}'")

index = pc.Index(INDEX_NAME)
print(f'\nIndex stats: {index.describe_index_stats()}')

In [ ]:
UPSERT_BATCH = 200

stats = index.describe_index_stats()
if stats.total_vector_count == 0:
    print(f'⏳ Upserting {len(passages):,} vectors...')
    for i in tqdm(range(0, len(passages), UPSERT_BATCH), desc='Upserting'):
        batch_p = passages[i:i+UPSERT_BATCH]
        batch_e = embeddings[i:i+UPSERT_BATCH]
        vectors = [
            {'id': p['id'], 'values': e,
             'metadata': {'text': p['text'][:1000], 'title': p['title']}}
            for p, e in zip(batch_p, batch_e)
        ]
        index.upsert(vectors=vectors)
    print(f"✅ Done! Total vectors: {index.describe_index_stats().total_vector_count:,}")
else:
    print(f'✅ Index already has {stats.total_vector_count:,} vectors, skipping upsert.')

## Step 6 — Load Cross-Encoder

`ms-marco-MiniLM-L-6-v2` is a lightweight cross-encoder trained on MS MARCO passage ranking.

In [ ]:
from sentence_transformers import CrossEncoder

print('⏳ Loading cross-encoder...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('✅ Cross-encoder ready!')

## Step 7 — Search Functions

In [ ]:
def semantic_search_no_rerank(query, top_k=TOP_K):
    """Bi-encoder only: embed query → Pinecone cosine similarity."""
    qvec = embed_query(query)
    res  = index.query(vector=qvec, top_k=top_k, include_metadata=True)
    return [
        {'rank': i+1, 'id': m['id'], 'title': m['metadata']['title'],
         'text': m['metadata']['text'], 'cosine_score': round(m['score'], 4)}
        for i, m in enumerate(res['matches'])
    ]


def semantic_search_with_rerank(query, top_k=TOP_K, top_n=TOP_N_RERANKED):
    """Bi-encoder retrieval + cross-encoder reranking."""
    # Stage 1: fast ANN retrieval
    qvec = embed_query(query)
    res  = index.query(vector=qvec, top_k=top_k, include_metadata=True)
    candidates = [
        {'id': m['id'], 'title': m['metadata']['title'],
         'text': m['metadata']['text'], 'cosine_score': round(m['score'], 4)}
        for m in res['matches']
    ]
    # Stage 2: cross-encoder reranking
    scores = cross_encoder.predict([[query, c['text']] for c in candidates]).tolist()
    for c, s in zip(candidates, scores):
        c['ce_score'] = round(s, 4)
    reranked = sorted(candidates, key=lambda x: x['ce_score'], reverse=True)[:top_n]
    for i, r in enumerate(reranked):
        r['rank'] = i + 1
    return reranked


print('✅ Search functions ready!')

## Step 8 — Side-by-Side Comparison

For each query we show bi-encoder results vs cross-encoder reranked results, plus a rank-shift analysis.

In [ ]:
import textwrap

def print_comparison(query):
    SEP = '═' * 82
    print(f'\n{SEP}')
    print(f'  🔍 QUERY: {query}')
    print(SEP)

    basic    = semantic_search_no_rerank(query, top_k=TOP_N_RERANKED)
    reranked = semantic_search_with_rerank(query, top_k=TOP_K, top_n=TOP_N_RERANKED)

    print(f'\n{"─"*35} BI-ENCODER ONLY {"─"*31}')
    print(f'  OpenAI/Local embedding → Pinecone cosine, top-{TOP_N_RERANKED}\n')
    for r in basic:
        snippet = textwrap.shorten(r['text'], width=110, placeholder='...')
        print(f"  [{r['rank']}] {r['title']:<35} cosine={r['cosine_score']}")
        print(f'      {snippet}\n')

    print(f'\n{"─"*30} BI-ENCODER + CROSS-ENCODER {"─"*25}')
    print(f'  Pinecone top-{TOP_K} → Cross-Encoder reranked to top-{TOP_N_RERANKED}\n')
    for r in reranked:
        snippet = textwrap.shorten(r['text'], width=110, placeholder='...')
        print(f"  [{r['rank']}] {r['title']:<35} cosine={r['cosine_score']}  ce={r['ce_score']}")
        print(f'      {snippet}\n')

    # Rank-shift analysis
    basic_ids    = [r['id'] for r in basic]
    reranked_ids = [r['id'] for r in reranked]
    shifts = sum(1 for i, rid in enumerate(reranked_ids) if i < len(basic_ids) and rid != basic_ids[i])
    if shifts:
        print(f'  ⚠️  Cross-encoder changed {shifts}/{TOP_N_RERANKED} positions!')
        if reranked_ids[0] != basic_ids[0]:
            old = (basic_ids.index(reranked_ids[0]) + 1) if reranked_ids[0] in basic_ids else '?'
            print(f'  🔺 New #1 was #{old} in bi-encoder ranking.')
    else:
        print('  ✅ Cross-encoder confirmed bi-encoder ranking — no changes.')
    print()


print('✅ Ready!')

### 🧪 Query 1 — History

In [ ]:
print_comparison('When did the First World War begin and what caused it?')

### 🧪 Query 2 — Science

In [ ]:
print_comparison('How does photosynthesis work in plants?')

### 🧪 Query 3 — Ambiguous short query (stress test for bi-encoder)

In [ ]:
print_comparison('capital punishment')

### 🧪 Query 4 — Geography

In [ ]:
print_comparison('What is the largest country in South America by area?')

## Step 9 — Interactive Search

In [ ]:
MY_QUERY = 'What were the main causes of the French Revolution?'
print_comparison(MY_QUERY)

## Step 10 — Latency Benchmark

In [ ]:
import timeit

BENCH_Q = 'What is the speed of light?'
N = 3

print('⏱️  Benchmarking (average of 3 runs)...\n')
t_basic   = timeit.timeit(lambda: semantic_search_no_rerank(BENCH_Q, top_k=TOP_N_RERANKED), number=N) / N
t_rerank  = timeit.timeit(lambda: semantic_search_with_rerank(BENCH_Q, top_k=TOP_K, top_n=TOP_N_RERANKED), number=N) / N
overhead  = t_rerank - t_basic

print(f'  🔵 Bi-encoder only           : {t_basic:.3f}s')
print(f'  🟠 Bi-encoder + Cross-encoder : {t_rerank:.3f}s')
print(f'  ⚖️  Reranking overhead         : +{overhead:.3f}s  ({overhead/t_basic*100:.0f}% slower)')
print()
print('💡 The cross-encoder only scores top-K candidates, so latency scales')
print('   with TOP_K (default 10), not with the full corpus size.')

## Step 11 — Score Distribution Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

VIZ_Q = 'Who invented the telephone?'

qvec    = embed_query(VIZ_Q)
raw     = index.query(vector=qvec, top_k=10, include_metadata=True)
cands   = [{'text': m['metadata']['text'], 'cosine': m['score']} for m in raw['matches']]
scores  = cross_encoder.predict([[VIZ_Q, c['text']] for c in cands]).tolist()
for c, s in zip(cands, scores):
    c['ce'] = s

cosines  = [c['cosine'] for c in cands]
ces      = [c['ce']     for c in cands]
labels   = [f'Doc {i+1}' for i in range(len(cands))]
ce_min, ce_max = min(ces), max(ces)
ces_norm = [(s - ce_min) / (ce_max - ce_min + 1e-9) for s in ces]

x, w = np.arange(len(labels)), 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Score Distribution — "{VIZ_Q}"', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(x - w/2, cosines,  w, label='Cosine (bi-encoder)',    color='steelblue',  alpha=0.85)
ax.bar(x + w/2, ces_norm, w, label='CE score (normalised)',  color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Bi-encoder vs Cross-Encoder Scores')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
sc  = ax2.scatter(cosines, ces_norm, c=range(len(cosines)), cmap='RdYlGn', s=120, zorder=3)
for i, lbl in enumerate(labels):
    ax2.annotate(lbl, (cosines[i], ces_norm[i]), textcoords='offset points', xytext=(5, 4), fontsize=8)
ax2.set_xlabel('Cosine similarity'); ax2.set_ylabel('Cross-encoder score (norm)')
ax2.set_title('Agreement: bi-encoder vs cross-encoder')
ax2.grid(alpha=0.3)
fig.colorbar(sc, ax=ax2, label='Pinecone rank (green=#1)')

plt.tight_layout()
plt.savefig('score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Dots far from the diagonal = docs where the two methods strongly disagree.')

## Step 12 — Cleanup (Optional)

In [ ]:
DELETE_INDEX = False   # set True to delete the Pinecone index

if DELETE_INDEX:
    pc.delete_index(INDEX_NAME)
    print(f"🗑️  Index '{INDEX_NAME}' deleted.")
else:
    print('Index kept. Flip DELETE_INDEX=True to remove it.')

---
## 📊 Summary: Bi-Encoder vs Cross-Encoder

| | Bi-Encoder Only | + Cross-Encoder |
|---|---|---|
| How it works | Query & doc embedded independently | Query & doc processed together |
| Speed | ⚡ Very fast (ANN lookup) | 🐢 Slower (runs on top-K only) |
| Accuracy | Good | Better — captures subtle relevance |
| Scalability | Millions of docs | top-K candidates only |
| Role in pipeline | Stage 1: retrieval | Stage 2: reranking |

The two-stage pipeline gives you the best of both worlds — Pinecone handles fast ANN search at scale, while the cross-encoder deeply re-scores just the most promising candidates.